# Apprentissage (Linear Regression)

### Données linéaires bruitées

In [ ]:
import torch
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(123)

In [ ]:
m = 50

X = 2 * torch.randn(m, 1)
y = 8 + 3 * X + torch.randn(m, 1)

## Exercice : Trouver à la main les paramètres theta qui fonctionne

In [ ]:
from ipywidgets import interact, FloatSlider

# Function to update plot
def plot_line(theta0, theta1):
    plt.figure(figsize=(8,5))
    plt.plot(X.numpy(), y.numpy(), "b.", label="Data")
    y_pred = theta0 + theta1 * X.numpy()
    plt.plot(X.numpy(), y_pred, "r-", linewidth=2, label=f"y = {theta0:.2f} + {theta1:.2f}x")
    plt.xlabel("$x$", fontsize=14)
    plt.ylabel("$y$", rotation=0, fontsize=14)
    plt.axis([0, 2, 0, 15])
    plt.legend()
    plt.show()

# Interactive sliders
interact(
    plot_line,
    theta0=FloatSlider(value=0, min=0, max=10, step=0.1),
    theta1=FloatSlider(value=0, min=0, max=10, step=0.1)
)

### Via Scikit-Learn (rappel)

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X, y)
lin_reg.intercept_, lin_reg.coef_

## Solution : Descente de Gradient

### Batch gradient descent

In [ ]:
m = X.shape[0]

In [ ]:
X_b = torch.hstack([X, torch.ones((m, 1))])

X_b.shape

In [ ]:
# Exercice 2

# Ajouter une colonne de 1 au vecteur X afin qu'on puisse le multiplier par theta -> Voir torch.hstack et torch.ones
# Définir : eta (learning rate) = 10 , nombre d'epochs = 10, thetas (initialisation) -> voir torch.randn
# Instancier une boucle où à chaque itération :
#   Calcul du gradient -> voir slide et torch.matmul ou l'opérateur @
#   Mise à jour de thetas -> voir slide


In [ ]:
# Suite exercice

# Réaliser une fonction permettant de visualiser les différentes droites trouvées à chaque epoch.
# Cette fonction doit avoir en argument eta
# Vous avez le droit de vous faire assister par IA

def plot_training_lines(X, y, eta=0.1, n_epochs=100):
    # TODO
    pass


plot_training_lines(X, y, eta=0.1, n_epochs=100)


In [ ]:
# Suite exercice

# Réaliser une fonction permettant d'afficher la courbe d'apprentissage.
# Bonus: split train/val afin de visualiser les deux courbes
# Vous avez le droit de vous faire assister par IA

import torch
import matplotlib.pyplot as plt

m = 100
X = 2 * torch.randn(m, 1)
y = 4 + 8 * X + torch.randn(m, 1)


def plot_learning_curve(X, y, eta=0.1, n_epochs=100, val_split=0.2):
    """
    Affiche la courbe d'apprentissage (erreur train/val) pour une régression linéaire
    avec descente de gradient.

    Args:
        X (torch.Tensor): Données d'entrée, shape (m,1)
        y (torch.Tensor): Cibles, shape (m,1)
        eta (float): Learning rate
        n_epochs (int): Nombre d'itérations
        val_split (float): Pourcentage de validation (0-1)
    """
    # TODO
    pass


plot_learning_curve(X, y, eta=0.1, n_epochs=20, val_split=0.1)


# Intérêt de la mise à l'échelle

In [ ]:
m = 100

X1 = 2 * torch.rand(m, 1)
X2 = 100 * torch.rand(m, 1)

X = torch.cat([X1, X2], dim=1)
X_scaled = torch.cat([(X1 - X1.mean(axis=0) )/ X1.std(axis=0), (X2 - X2.mean(axis=0) )/ X2.std(axis=0)], dim=1)

# Biais = 15.5, Poids de X1 = 2, Poids de X2 = 5.6
y = 15.5 + 2 * X1 + 5.6 * X2

# Faites tourner la fonction précédente sur ces nouvelles données,
# puis comparez avec les mêmes données mises à l’échelle (StandardScaler).
# Que remarquez-vous ?

In [ ]:
plot_learning_curve(X, y, eta=3e-1, n_epochs=50, val_split=0.1)

In [ ]:
thetas = plot_learning_curve(X_scaled, y, eta=3e-1, n_epochs=10, val_split=0.1)

Le modèle entraîné sur `X_scaled` a trouvé cette équation :

$$ y = \theta'_1 \left( \frac{X_1 - \mu_1}{\sigma_1} \right) + \theta'_2 \left( \frac{X_2 - \mu_2}{\sigma_2} \right) + \theta'_{biais} $$

Si on développe pour isoler $X_1$ et $X_2$, on obtient l'équation sur les variables d'origine :

$$ y = \left( \frac{\theta'_1}{\sigma_1} \right) X_1 + \left( \frac{\theta'_2}{\sigma_2} \right) X_2 + \left( \theta'_{biais} - \frac{\theta'_1 \mu_1}{\sigma_1} - \frac{\theta'_2 \mu_2}{\sigma_2} \right) $$

In [ ]:
# Pour passer des paramètres trouvés sur les données mise à l'échelle à ceux d'origine il faut : 
mu1, std1 = X1.mean().item(), X1.std().item()
mu2, std2 = X2.mean().item(), X2.std().item()

theta1_scaled = thetas[0]    # Poids pour X1_scaled
theta2_scaled = thetas[1] # Poids pour X2_scaled
bias_scaled   = thetas[2]  # Biais_scaled (dernier élément de thetas)

# 1. Conversion des poids
theta1_orig = theta1_scaled / std1
theta2_orig = theta2_scaled / std2

# 2. Conversion du biais
bias_orig = bias_scaled - (theta1_orig * mu1) - (theta2_orig * mu2)

print(f"Poids d'origine reconstitués :")
print(f"Poids X1 : {theta1_orig.item():.2f} (Attendu : ~2.0)")
print(f"Poids X2 : {theta2_orig.item():.2f} (Attendu : ~5.6)")
print(f"Biais    : {bias_orig.item():.2f} (Attendu : ~15.5)")

### Stochastic Gradient Descent

In [ ]:
# Exercice : Adapter la batch gradient descente à la stochastic gradient descent

m = 100
X = 5 * torch.randn(m, 1)
y = 4 + 8 * X + torch.randn(m, 1)
X_b = torch.hstack([X, torch.ones((m, 1))])

eta = 0.01
n_epochs = 100

thetas = torch.randn(2, 1)
# TODO : boucle SGD
#  - à chaque epoch, mélanger les indices (torch.randperm)
#  - mettre à jour thetas instance par instance


In [ ]:
def plot_learning_curve_SGD(X, y, eta=0.1, n_epochs=100, val_split=0.2):
    """
    Affiche la courbe d'apprentissage (erreur train/val) pour une régression linéaire
    avec descente de gradient stochastique (SGD).

    Args:
        X (torch.Tensor): Données d'entrée, shape (m, n)
        y (torch.Tensor): Cibles, shape (m, 1)
        eta (float): Learning rate
        n_epochs (int): Nombre d'itérations sur tout le dataset
        val_split (float): Pourcentage de validation (0-1)
    """
    # TODO
    pass


plot_learning_curve_SGD(X, y, eta=0.01, n_epochs=20, val_split=0.1)


### Mini-Batch Gradient Descent

In [ ]:
# Exerice : Adapter la stochastic gradient descente à la mini-batch gradient descent

batch_size = 32

thetas = torch.randn(2, 1)
# TODO : boucle mini-batch
#  - à chaque epoch, mélanger les indices
#  - mettre à jour thetas batch par batch (batch_size)


In [ ]:
def plot_learning_curve_MiniBGD(X, y, eta=0.1, n_epochs=100, val_split=0.2, batch_size=32):
    """
    Affiche la courbe d'apprentissage (erreur train/val) pour une régression linéaire
    avec descente de gradient mini-batch.

    Args:
        X (torch.Tensor): Données d'entrée, shape (m, n)
        y (torch.Tensor): Cibles, shape (m, 1)
        eta (float): Learning rate
        n_epochs (int): Nombre d'itérations sur tout le dataset
        val_split (float): Pourcentage de validation (0-1)
        batch_size (int): Taille des mini-batchs
    """
    # TODO
    pass


plot_learning_curve_MiniBGD(X, y, eta=0.01, n_epochs=20, val_split=0.1)


# Logistic Regression

In [ ]:
from sklearn.datasets import load_breast_cancer
import torch
X, y = load_breast_cancer(as_frame=True, return_X_y=True)

In [ ]:
X

In [ ]:
import numpy as np
np.unique(y, return_counts=True)

In [ ]:
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32)

In [ ]:
X_b = torch.hstack([(X - X.mean(axis=0)) / X.std(axis=0), torch.ones(X.shape[0], 1)])

X_b.shape

In [ ]:
# Exercice : Modifier la linear regression pour réaliser de la logistic regression
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

m = X_b.shape[0]
n_epochs = 50
eta = 0.1
batch_size = 16
num_features = X_b.shape[1]
thetas = torch.randn(num_features, 1)
# TODO : boucle d'entraînement mini-batch de la régression logistique
#  - passer la sortie linéaire dans sigmoid
#  - calculer le gradient et mettre à jour thetas


In [ ]:
(((X_b @ thetas > 0) * 1.0 == y[..., None]) * 1.0).mean() * 100

# Softmax Regression

In [ ]:
from sklearn.datasets import load_iris
import torch

X, y = load_iris(as_frame=True, return_X_y=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.int64).to(device)

X_b = torch.hstack([(X - X.mean(axis=0)) / X.std(axis=0), torch.ones(X.shape[0], 1)]).to(device)
X_b.shape

In [ ]:
y

In [ ]:
num_features = X_b.shape[1]
num_classes = torch.unique(y).shape[0]

In [ ]:
y = torch.nn.functional.one_hot(y, num_classes=num_classes).float()

In [ ]:
def softmax(x):
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum(axis=1)[..., None]

batch_size = 16
eta = 1e-1
n_epochs = 100
m = X_b.shape[0]
thetas = torch.randn(num_features, num_classes).to(device)
# TODO : boucle d'entraînement mini-batch de la régression softmax
#  - passer la sortie linéaire dans softmax
#  - calculer le gradient et mettre à jour thetas
#  - afficher l'accuracy à chaque epoch


### Exercice — Effet du *learning rate* sur la convergence
Implementez une descente de gradient batch pour une régression linéaire simple et comparez la convergence pour plusieurs learning rates. Un LR trop grand **diverge**, un LR trop petit **converge lentement**.

Faites le sur n'importe quel algorithme et avec n'importe quelle données


In [ ]:
# Exercice : effet du learning rate sur la convergence (batch gradient descent)

import torch
import matplotlib.pyplot as plt

torch.manual_seed(123)

# --- Données linéaires bruitées ---
m = 100
X = 2 * torch.randn(m, 1)
y = 4 + 8 * X + torch.randn(m, 1)

# Ajout du biais
X_b = torch.hstack([X, torch.ones((m, 1))])

# TODO :
#  - écrire une descente de gradient batch qui retourne l'historique des MSE
#  - comparer plusieurs learning rates (ex: [0.001, 0.01, 0.1, 0.5])
#  - tracer log10(MSE) en fonction des epochs pour visualiser
#    la convergence lente (eta trop petit) et la divergence (eta trop grand)
